# Dynamic Withdrawal Strategies - Step-by-Step Test

이 노트북은 Dynamic 전략 구현을 단계별로 테스트합니다.

## Step 1: 라이브러리 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 로드 완료")

In [ ]:
# 벤치마크 데이터 로드
with open('benchmark_data.pkl', 'rb') as f:
    data = pickle.load(f)

print(f"✅ 벤치마크 데이터 로드 완료")
print(f"   데이터 크기: {data.shape}")
print(f"   날짜 범위: {data.index[0].date()} ~ {data.index[-1].date()}")
print(f"\n컬럼:\n{data.columns.tolist()}")

## Step 2: DataPreprocessor 테스트

In [ ]:
from withdrawal_backtest import DataPreprocessor, PORTFOLIOS

print(f"DataPreprocessor 임포트 성공")
print(f"\n포트폴리오 목록:")
for i, port_name in enumerate(PORTFOLIOS.keys(), 1):
    port = PORTFOLIOS[port_name]
    print(f"  {i}. {port_name}: 목표 수익률={port['target_return']:.1f}%, 목표 변동성={port['target_risk']:.2f}%")

In [ ]:
# DataPreprocessor 실행
try:
    preprocessor = DataPreprocessor(data, add_portfolios=True)
    returns_df, month_starts = preprocessor.get_data()
    
    print(f"✅ DataPreprocessor 완료")
    print(f"   Returns DataFrame: {returns_df.shape}")
    print(f"   Month Starts Series: {month_starts.shape}")
    print(f"\n   Returns 컬럼:")
    print(f"   {returns_df.columns.tolist()[:10]}...")
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 3: Dynamic Simulator 테스트

In [ ]:
from dynamic_simulator import GuardrailsWithdrawal, GuytonKlingerWithdrawal, DynamicWithdrawalSimulator

print("✅ Dynamic Simulator 클래스 임포트 성공")

# DynamicWithdrawalSimulator 생성
simulator = DynamicWithdrawalSimulator(returns_df, month_starts)
print(f"✅ DynamicWithdrawalSimulator 생성 완료")
print(f"   총 날짜: {len(simulator.dates)}")
print(f"   월초 개수: {np.sum(month_starts)}")

## Step 4: Guardrails 백테스트 (간단한 테스트)

In [ ]:
# 하나의 포트폴리오와 인출률로 빠른 테스트
test_portfolio = 'Port_5.0%'
test_wr = 0.05  # 5%
horizon_years = 10

print(f"테스트 설정:")
print(f"  포트폴리오: {test_portfolio}")
print(f"  인출률: {test_wr*100:.1f}%")
print(f"  기간: {horizon_years}년")

try:
    results_df = simulator.run_guardrails_backtest(
        benchmark=test_portfolio,
        horizon_years=horizon_years,
        initial_wr=test_wr,
        guardrail_width=0.20,
        inflation_rate=0.02,
        v0=100.0,
        verbose=True
    )
    
    print(f"\n✅ Guardrails 백테스트 완료")
    print(f"\n결과 DataFrame:")
    print(f"  크기: {results_df.shape}")
    print(f"  컬럼: {results_df.columns.tolist()}")
    print(f"\n샘플 데이터 (첫 5행):")
    display(results_df[['start_date', 'terminal_nav', 'total_withdrawal', 'is_fail']].head())
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 5: 메트릭 계산 테스트

In [ ]:
from metrics_calculator import MetricsCalculator

metrics_calc = MetricsCalculator()

try:
    metrics = metrics_calc.calculate_optimization_metrics(results_df, v0=100.0)
    
    print(f"✅ 메트릭 계산 완료")
    print(f"\n주요 메트릭:")
    print(f"  총 인출액 (평균): {metrics['total_withdrawal_mean']:,.2f}")
    print(f"  총 인출액 (범위): {metrics['total_withdrawal_worst']:,.2f} ~ {metrics['total_withdrawal_best']:,.2f}")
    print(f"  YoY 변동성 (평균): {metrics['yoy_volatility_mean']:.4f}")
    print(f"  YoY 변동성 (90분위): {metrics['yoy_volatility_90pct']:.4f}")
    print(f"  실패율: {metrics['failure_rate']:.2%}")
    print(f"  최종 NAV (중앙값): {metrics['terminal_nav_median']:.2%}")
    print(f"  경로 수: {metrics['total_paths']}")
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 6: Guyton-Klinger 백테스트

In [ ]:
try:
    gk_results_df = simulator.run_guyton_klinger_backtest(
        benchmark=test_portfolio,
        horizon_years=horizon_years,
        initial_wr=test_wr,
        guardrail_width=0.20,
        adjustment_pct=0.10,
        freeze_threshold=-0.10,
        inflation_rate=0.02,
        v0=100.0,
        verbose=True
    )
    
    print(f"\n✅ Guyton-Klinger 백테스트 완료")
    print(f"\n결과 DataFrame:")
    print(f"  크기: {gk_results_df.shape}")
    print(f"\n샘플 데이터 (첫 5행):")
    display(gk_results_df[['start_date', 'terminal_nav', 'total_withdrawal', 'is_fail']].head())
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 6.1: Guyton-Klinger 일별 Path 값 조회

`gk_results_df`의 `withdrawal_path`/`nav_path`는 월별 배열이므로, `get_single_path_detail`을 사용하여 일별 경로 데이터를 조회합니다.

In [ ]:
# ============================================================
# 시작일 설정 - 원하는 날짜로 변경하세요
# ============================================================
start_date = '2008-01-02'  # 원하는 시작일로 변경

# 자산 표시명 → 원본 data 컬럼명 매핑 (가격 지수용)
PRICE_COL_MAP = {
    'Korean_Equity':          '국내주식',
    'US_Growth':              '미국성장주',
    'Korean_Bond_Composite':  '국내중기채',
    'Korean_Bond_10Y':        '국내장기채',
    'EM_Dollar_Bond':         '신흥국달러채권',
    'US_Bond':                '미국국채',
    'Global_ex_US_Bond':      '미국외국채',
    'Gold':                   '금',
}

# gk_results_df에서 해당 start_date의 월별 경로 먼저 확인
row = gk_results_df[gk_results_df['start_date'] == pd.Timestamp(start_date)]
if row.empty:
    # 가장 가까운 날짜 찾기
    available_dates = gk_results_df['start_date'].sort_values()
    print(f"'{start_date}'는 유효한 시작일이 아닙니다.")
    print(f"사용 가능한 시작일 범위: {available_dates.iloc[0].date()} ~ {available_dates.iloc[-1].date()}")
    print(f"\n가장 가까운 시작일 5개:")
    target = pd.Timestamp(start_date)
    closest = available_dates.iloc[(available_dates - target).abs().argsort()[:5]]
    for d in closest:
        print(f"  {d.date()}")
else:
    row = row.iloc[0]
    print(f"=== gk_results_df 월별 경로 (start_date: {start_date}) ===")
    print(f"Terminal NAV: {row['terminal_nav']:.2f}")
    print(f"Total Withdrawal: {row['total_withdrawal']:.2f}")
    print(f"Is Fail: {row['is_fail']}")
    
    # 월별 nav_path / withdrawal_path
    nav_monthly = row['nav_path']
    wd_monthly = row['withdrawal_path']
    print(f"\n월별 NAV path (길이={len(nav_monthly)}):")
    monthly_df = pd.DataFrame({
        'Month': range(len(nav_monthly)),
        'NAV': nav_monthly,
        'Withdrawal': wd_monthly
    })
    display(monthly_df)
    
    # ============================================================
    # 일별 경로 조회 (get_single_path_detail)
    # ============================================================
    print(f"\n{'='*60}")
    print(f"일별 경로 데이터 조회 (start_date: {start_date})")
    print(f"{'='*60}")
    
    daily_path_df = simulator.get_single_path_detail(
        portfolio=test_portfolio,
        start_date=start_date,
        strategy='guyton_klinger',
        horizon_years=horizon_years,
        initial_wr=test_wr,
        guardrail_width=0.20,
        adjustment_pct=0.10,
        freeze_threshold=-0.10,
        inflation_rate=0.02,
        v0=100.0
    )
    
    # ============================================================
    # 자산별 Price Index Level 추가
    # ============================================================
    dates_in_path = daily_path_df['Date'].values
    for display_name, data_col in PRICE_COL_MAP.items():
        col_name = f'Price_{display_name}'
        daily_path_df[col_name] = data.loc[dates_in_path, data_col].values
    
    print(f"\n일별 경로 DataFrame (Price Index 포함):")
    print(f"  크기: {daily_path_df.shape}")
    print(f"  컬럼: {daily_path_df.columns.tolist()}")
    print(f"  날짜 범위: {daily_path_df['Date'].iloc[0].date()} ~ {daily_path_df['Date'].iloc[-1].date()}")
    print(f"  총 거래일: {len(daily_path_df)}")
    
    # NAV 컬럼과 Price 컬럼 나란히 표시
    nav_cols = [c for c in daily_path_df.columns if c.startswith('NAV_')]
    price_cols = [c for c in daily_path_df.columns if c.startswith('Price_')]
    display_cols = ['Date', 'Total_NAV'] + nav_cols + price_cols + [
        'Monthly_Withdrawal', 'Is_Month_Start', 'Guardrail_Status'
    ]
    
    print(f"\n--- 일별 Total NAV, 자산별 NAV & Price Index ---")
    display(daily_path_df[display_cols])
    
    # 월초만 필터링하여 백테스트 경로와 비교
    month_start_rows = daily_path_df[daily_path_df['Is_Month_Start'] == True].copy()
    print(f"\n--- 월초 기준 일별 경로 vs 백테스트 월별 경로 비교 ---")
    print(f"  월초 행 수: {len(month_start_rows)}")
    comparison = pd.DataFrame({
        'Date': month_start_rows['Date'].values,
        'Daily_NAV': month_start_rows['Total_NAV'].values[:len(nav_monthly)],
        'Monthly_NAV': nav_monthly[:len(month_start_rows)],
        'Daily_WD': month_start_rows['Monthly_Withdrawal'].values[:len(wd_monthly)],
        'Monthly_WD': wd_monthly[:len(month_start_rows)]
    })
    comparison['NAV_Diff'] = abs(comparison['Daily_NAV'] - comparison['Monthly_NAV'])
    comparison['WD_Diff'] = abs(comparison['Daily_WD'] - comparison['Monthly_WD'])
    display(comparison)

## Step 6.5: Fixed Rate 백테스트

In [ ]:
try:
    fixed_results_df = simulator.run_fixed_backtest(
        benchmark=test_portfolio,
        horizon_years=horizon_years,
        initial_wr=test_wr,
        inflation_rate=0.02,
        v0=100.0,
        verbose=True
    )
    
    print(f"\n✅ Fixed Rate 백테스트 완료")
    print(f"\n결과 DataFrame:")
    print(f"  크기: {fixed_results_df.shape}")
    print(f"\n샘플 데이터 (첫 5행):")
    display(fixed_results_df[['start_date', 'terminal_nav', 'total_withdrawal', 'is_fail']].head())
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 7: WithdrawalOptimizer 테스트

In [ ]:
from optimizer import WithdrawalOptimizer

optimizer = WithdrawalOptimizer(returns_df, month_starts)

print(f"✅ WithdrawalOptimizer 생성 완료")

# 작은 범위로 빠른 테스트
withdrawal_rates = np.array([0.04, 0.05, 0.06])
constraints = {
    'max_yoy_volatility': 0.20,
    'max_failure_rate': 0.10,
    'min_terminal_nav': 0.40
}

print(f"\n테스트 설정:")
print(f"  인출률: {withdrawal_rates}")
print(f"  제약조건:")
print(f"    - Max YoY Volatility: {constraints['max_yoy_volatility']:.0%}")
print(f"    - Max Failure Rate: {constraints['max_failure_rate']:.0%}")
print(f"    - Min Terminal NAV: {constraints['min_terminal_nav']:.0%}")

In [ ]:
try:
    # 단일 포트폴리오 최적화 (빠른 테스트)
    portfolio_results = optimizer.optimize_single_portfolio(
        portfolio_name='Port_5.0%',
        withdrawal_rates=withdrawal_rates,
        horizon_years=10,
        constraints=constraints,
        v0=100.0,
        verbose=True
    )
    
    print(f"\n✅ 단일 포트폴리오 최적화 완료")
    print(f"\n결과 구조:")
    for wr, strategies in portfolio_results.items():
        print(f"  인출률 {wr:.2%}:")
        for strategy, metrics in strategies.items():
            feasible = "✓" if metrics.get('is_feasible') else "✗"
            total_w = metrics.get('total_withdrawal_mean', 0)
            print(f"    {strategy:20s} {feasible} - 총 인출액: {total_w:,.2f}")
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 8: 전체 최적화 (2개 포트폴리오)

In [ ]:
try:
    # 작은 범위로 빠른 테스트
    test_portfolios = ['Port_4.0%', 'Port_5.0%']
    test_wrs = np.array([0.04, 0.05, 0.06])
    
    results_df = optimizer.optimize_all_portfolios(
        portfolio_names=test_portfolios,
        withdrawal_rates=test_wrs,
        horizon_years=10,
        constraints=constraints,
        v0=100.0
    )
    
    print(f"\n✅ 전체 최적화 완료")
    print(f"\n결과 DataFrame: {results_df.shape}")
    print(f"\n전체 결과:")
    display(results_df.head(10))
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 9: 결과 요약

In [ ]:
try:
    print(f"\n최적화 결과 통계:")
    print(f"  전체 시나리오: {len(results_df)}")
    print(f"  가능한 시나리오 (제약 만족): {len(results_df[results_df['Feasible']==True])}")
    print(f"  불가능한 시나리오: {len(results_df[results_df['Feasible']==False])}")
    
    print(f"\n포트폴리오별 최적 솔루션:")
    for portfolio in results_df['Portfolio'].unique():
        df_port = results_df[results_df['Portfolio'] == portfolio]
        for strategy in ['fixed', 'guardrails', 'guyton_klinger']:
            df_strat = df_port[df_port['Strategy'] == strategy]
            df_feasible = df_strat[df_strat['Feasible'] == True]
            if not df_feasible.empty:
                best = df_feasible.loc[df_feasible['Total_Withdrawal'].idxmax()]
                print(f"  {portfolio:12s} {strategy:20s}: WR={best['WR']:.2%}, "
                      f"TotalW={best['Total_Withdrawal']:.0f}, "
                      f"Vol={best['YoY_Volatility']:.2%}")
            else:
                print(f"  {portfolio:12s} {strategy:20s}: No feasible solution")
                
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 10: UI 컴포넌트 테스트

In [ ]:
try:
    from dynamic_strategy_ui import (
        create_pareto_frontier_chart,
        create_strategy_comparison_chart
    )
    
    print(f"✅ UI 컴포넌트 임포트 성공")
    
    # Pareto Frontier 차트 생성
    portfolio_chart = 'Port_5.0%'
    fig = create_pareto_frontier_chart(results_df, portfolio_chart)
    
    print(f"✅ Pareto Frontier 차트 생성 완료")
    print(f"   차트 타입: {type(fig)}")
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 11: 특정 경로 상세 조회 (get_single_path_detail)

단일 시작일의 일별 경로 데이터를 반환하는 기능 테스트 (3가지 전략, 8개 자산 NAV 추적)

In [ ]:
# This cell is no longer needed - functionality merged into Step 11

In [ ]:
from dynamic_simulator import validate_backtest_consistency

print("\n" + "="*60)
print("백테스트 vs single_path_detail 일관성 검증")
print("="*60)

# 검증할 전략 및 백테스트 결과
strategies_to_validate = [
    ('fixed', fixed_results_df),
    ('guardrails', results_df),
    ('guyton_klinger', gk_results_df)
]

for strategy_name, backtest_df in strategies_to_validate:
    print(f"\n{'='*60}")
    print(f"{strategy_name.upper()} 전략 검증")
    print(f"{'='*60}")
    
    try:
        if strategy_name == 'fixed':
            result = validate_backtest_consistency(
                simulator=simulator,
                backtest_results=backtest_df,
                portfolio=test_portfolio,
                strategy='fixed',
                horizon_years=horizon_years,
                initial_wr=test_wr,
                inflation_rate=0.02,
                v0=100.0,
                sample_size=3,
                tolerance=1e-4
            )
        elif strategy_name == 'guardrails':
            result = validate_backtest_consistency(
                simulator=simulator,
                backtest_results=backtest_df,
                portfolio=test_portfolio,
                strategy='guardrails',
                horizon_years=horizon_years,
                initial_wr=test_wr,
                guardrail_width=0.20,
                inflation_rate=0.02,
                v0=100.0,
                sample_size=3,
                tolerance=1e-4
            )
        else:  # guyton_klinger
            result = validate_backtest_consistency(
                simulator=simulator,
                backtest_results=backtest_df,
                portfolio=test_portfolio,
                strategy='guyton_klinger',
                horizon_years=horizon_years,
                initial_wr=test_wr,
                guardrail_width=0.20,
                adjustment_pct=0.10,
                freeze_threshold=-0.10,
                inflation_rate=0.02,
                v0=100.0,
                sample_size=3,
                tolerance=1e-4
            )
        
        if result['all_passed']:
            print(f"✅ 모든 검증 통과: {result['n_passed']}/{result['n_tested']}")
        else:
            print(f"❌ 일부 검증 실패: {result['n_passed']}/{result['n_tested']}")
            print(f"\n실패한 케이스:")
            for case in result['failed_cases']:
                print(f"  - 경로 {case['idx']}: {case['error']}")
                
    except Exception as e:
        print(f"❌ 검증 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*60}")
print(f"전체 일관성 검증 완료")
print(f"{'='*60}")

In [ ]:
# Guyton-Klinger 테스트
try:
    path_df_gk = simulator.get_single_path_detail(
        portfolio=test_portfolio,
        start_date=test_start_date,
        strategy='guyton_klinger',
        horizon_years=10,
        initial_wr=0.05,
        guardrail_width=0.20,
        adjustment_pct=0.10,
        freeze_threshold=-0.10,
        inflation_rate=0.02,
        v0=100.0
    )
    
    print(f"\u2705 Guyton-Klinger 일별 경로 데이터 생성 완료")
    print(f"  크기: {path_df_gk.shape}")
    print(f"  날짜 범위: {path_df_gk['Date'].iloc[0].date()} ~ {path_df_gk['Date'].iloc[-1].date()}")
    
    # 검증
    path_df_gk['Calculated_Total'] = (
        path_df_gk['NAV_Korean_Equity'] + 
        path_df_gk['NAV_US_Growth'] + 
        path_df_gk['NAV_Bond'] + 
        path_df_gk['NAV_Gold']
    )
    max_diff_gk = abs(path_df_gk['Total_NAV'] - path_df_gk['Calculated_Total']).max()
    print(f"  Total_NAV 최대 오차: {max_diff_gk:.10f}")
    
    print(f"  Guardrail 상태 분포:")
    print(path_df_gk['Guardrail_Status'].value_counts())
    
    # 엑셀 저장
    output_path_gk = f'path_{test_start_date}_guyton_klinger.xlsx'
    path_df_gk.drop(columns=['Calculated_Total'], inplace=True)
    path_df_gk.to_excel(output_path_gk, index=False)
    print(f"\u2705 엑셀 저장 완료: {output_path_gk}")
    
except Exception as e:
    print(f"\u274c 오류: {e}")
    import traceback
    traceback.print_exc()